# 🚀 Building RAG from Scratch - Learning Notebook

**Learn Retrieval-Augmented Generation (RAG) Step by Step**

This notebook teaches you how to build a RAG system from the ground up using only basic Python libraries:
- **OpenAI** for embeddings and text generation
- **NumPy** for vector similarity calculations  
- **JSON** for simple vector database storage

## 📚 What You'll Learn:

1. **Document Processing** - Loading and converting JSON to text
2. **Text Chunking** - Breaking documents into manageable pieces  
3. **Embeddings** - Converting text to numerical vectors
4. **Vector Search** - Finding similar content using cosine similarity
5. **Answer Generation** - Using AI to create contextual responses
6. **Transparent Outputs** - Saving all intermediate results to files

## 🎯 Goal: 
Build a working RAG system that can answer questions about the **Flintstone API** documentation!

---

## 📦 Step 1: Import Libraries & Configuration

First, let's import all the libraries we need and set up our configuration.

In [26]:
import json
import os
import numpy as np
from typing import List, Dict, Any
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# ✅ Configuration - All settings in one place
class RAGConfig:
    # OpenAI Settings
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
    GENERATION_MODEL = "gpt-4o-mini" 
    EMBEDDING_MODEL = "text-embedding-ada-002"
    TEMPERATURE = 0.1
    MAX_TOKENS = 2000
    
    # Document Processing
    CHUNK_SIZE = 1000
    CHUNK_OVERLAP = 200
    
    # Files
    INPUT_DOCUMENT = "./documents/flintstone_api.json"
    VECTOR_DATABASE_FILE = "vector_database.json"
    
    # RAG Settings
    TOP_K_RESULTS = 3

# Initialize OpenAI client
if not RAGConfig.OPENAI_API_KEY:
    raise ValueError("❌ Please set OPENAI_API_KEY in your .env file")

client = OpenAI(api_key=RAGConfig.OPENAI_API_KEY)

print("✅ Libraries imported and configuration loaded!")
print(f"📁 Input document: {RAGConfig.INPUT_DOCUMENT}")
print(f"🗄️ Vector database: {RAGConfig.VECTOR_DATABASE_FILE}")
print(f"🔢 Chunk size: {RAGConfig.CHUNK_SIZE} characters")
print(f"🔍 Retrieving top {RAGConfig.TOP_K_RESULTS} results")

✅ Libraries imported and configuration loaded!
📁 Input document: ./documents/flintstone_api.json
🗄️ Vector database: vector_database.json
🔢 Chunk size: 1000 characters
🔍 Retrieving top 3 results


## 📄 Step 2: Document Loading & Processing 

Let's load our Flintstone API JSON file and convert it to readable text for better embedding quality.

In [5]:
def json_to_text(data, indent=0):
    """
    🔄 Convert JSON to readable text format
    This helps with better chunking and embedding quality
    """
    result = []
    prefix = "  " * indent
    
    if isinstance(data, dict):
        for key, value in data.items():
            result.append(f"{prefix}{key}:")
            if isinstance(value, (dict, list)):
                result.append(json_to_text(value, indent + 1))
            else:
                result.append(f"{prefix}  {value}")
    elif isinstance(data, list):
        for item in data:
            if isinstance(item, (dict, list)):
                result.append(json_to_text(item, indent))
            else:
                result.append(f"{prefix}- {item}")
    
    return "\n".join(result)

print("✅ JSON-to-text conversion function ready!")

✅ JSON-to-text conversion function ready!


In [6]:
# 📖 Load the JSON document
print(f"📖 Loading document: {RAGConfig.INPUT_DOCUMENT}")

with open(RAGConfig.INPUT_DOCUMENT, 'r', encoding='utf-8') as file:
    raw_json_data = json.load(file)

print(f"✅ JSON document loaded!")
print(f"📊 Document keys: {list(raw_json_data.keys())}")
print(f"🔢 Total JSON size: {len(str(raw_json_data))} characters")

📖 Loading document: ./documents/flintstone_api.json
✅ JSON document loaded!
📊 Document keys: ['api_name', 'version', 'description', 'base_url', 'authentication', 'rate_limits', 'endpoints', 'error_codes', 'response_formats', 'pagination', 'changelog', 'contact']
🔢 Total JSON size: 11245 characters


In [7]:
# 🔄 Convert JSON to readable text
print("🔄 Converting JSON to readable text...")

document_text = json_to_text(raw_json_data)

print(f"✅ Conversion complete!")
print(f"📏 Text length: {len(document_text)} characters")

# Show a preview
print("\n📋 Preview of converted text (first 300 characters):")
print("-" * 50)
print(document_text[:300] + "..." if len(document_text) > 300 else document_text)

🔄 Converting JSON to readable text...
✅ Conversion complete!
📏 Text length: 14974 characters

📋 Preview of converted text (first 300 characters):
--------------------------------------------------
api_name:
  Flintstone API
version:
  2.1.0
description:
  The comprehensive API for managing prehistoric Bedrock operations, citizens, and resources
base_url:
  https://api.bedrock.stone
authentication:
  type:
    stone_token
  header:
    X-Stone-Token
  description:
    Carved stone tokens requi...


In [8]:
# 💾 Save converted text for inspection
with open("converted_text_output.txt", "w", encoding='utf-8') as f:
    f.write("=" * 60 + "\n")
    f.write("CONVERTED JSON TO TEXT OUTPUT\n")
    f.write(f"Source: {RAGConfig.INPUT_DOCUMENT}\n")
    f.write("=" * 60 + "\n\n")
    f.write(document_text)

print("💾 Converted text saved to: converted_text_output.txt")
print("🔍 You can now inspect this file to see the full conversion!")

💾 Converted text saved to: converted_text_output.txt
🔍 You can now inspect this file to see the full conversion!


## ✂️ Step 3: Text Chunking

Now let's break our document into smaller chunks for better retrieval. We'll use overlapping chunks to maintain context.

In [9]:
def chunk_text(text: str) -> List[str]:
    """
    ✂️ Split text into overlapping chunks
    
    Why overlapping? To maintain context between chunks and avoid
    cutting important information in half.
    """
    chunks = []
    start = 0
    
    while start < len(text):
        # Get chunk of specified size
        end = start + RAGConfig.CHUNK_SIZE
        chunk = text[start:end]
        
        # If this isn't the last chunk, try to break at word boundary
        if end < len(text):
            last_space = chunk.rfind(' ')
            if last_space > 0:
                chunk = chunk[:last_space]
                end = start + last_space
        
        chunks.append(chunk.strip())
        
        # Move start position with overlap
        start = end - RAGConfig.CHUNK_OVERLAP
        
        # Prevent infinite loop if chunk overlap is too large
        if start >= end:
            start = end
    
    return chunks

print("✅ Text chunking function ready!")

✅ Text chunking function ready!


In [10]:
# ✂️ Create chunks from the document text
print("✂️ Chunking the document...")

text_chunks_list = chunk_text(document_text)

print(f"✅ Created {len(text_chunks_list)} chunks")
print(f"📊 Average chunk size: {sum(len(c) for c in text_chunks_list) // len(text_chunks_list)} characters")

# Show info about first few chunks
print(f"\n📋 First 3 chunks info:")
for i, chunk in enumerate(text_chunks_list[:3]):
    print(f"  Chunk {i+1}: {len(chunk)} characters - '{chunk[:50]}...'")

✂️ Chunking the document...
✅ Created 19 chunks
📊 Average chunk size: 969 characters

📋 First 3 chunks info:
  Chunk 1: 985 characters - 'api_name:
  Flintstone API
version:
  2.1.0
descri...'
  Chunk 2: 993 characters - 'ring (optional) - Filter by job (quarry_worker, di...'
  Chunk 3: 990 characters - '-Dabba-Doo!
            bowling_average:
         ...'


In [11]:
# 📋 Add metadata to chunks for tracking
chunks_with_metadata = []
for i, chunk in enumerate(text_chunks_list):
    chunk_data = {
        "id": i,
        "text": chunk,
        "length": len(chunk),
        "start_pos": i * (RAGConfig.CHUNK_SIZE - RAGConfig.CHUNK_OVERLAP)
    }
    chunks_with_metadata.append(chunk_data)

print(f"📋 Created metadata for {len(chunks_with_metadata)} chunks")
print(f"🔍 Each chunk has: id, text, length, start_pos")

📋 Created metadata for 19 chunks
🔍 Each chunk has: id, text, length, start_pos


In [13]:
# 💾 Save chunks for inspection (FULL CONTENT - NO TRUNCATION)
with open("text_chunks_output.txt", "w", encoding='utf-8') as f:
    f.write("=" * 60 + "\n")
    f.write("TEXT CHUNKS OUTPUT\n")
    f.write(f"Total chunks: {len(chunks_with_metadata)}\n")
    f.write(f"Chunk size: {RAGConfig.CHUNK_SIZE} characters\n")
    f.write(f"Overlap: {RAGConfig.CHUNK_OVERLAP} characters\n")
    f.write("=" * 60 + "\n\n")
    
    for chunk_data in chunks_with_metadata:
        f.write(f"CHUNK {chunk_data['id'] + 1}:\n")
        f.write(f"ID: {chunk_data['id']}\n")
        f.write(f"Length: {chunk_data['length']} characters\n")
        f.write(f"Position: {chunk_data['start_pos']}-{chunk_data['start_pos'] + chunk_data['length']}\n")
        f.write("-" * 40 + "\n")
        # 🔥 SHOW FULL CHUNK CONTENT - NO TRUNCATION!
        f.write(chunk_data['text'])
        f.write("\n" + "=" * 60 + "\n\n")

print("💾 Chunks saved to: text_chunks_output.txt")
print("🔍 You can inspect this file to see ALL FULL chunks in complete detail!")
print("✅ NO TRUNCATION - Every chunk shows its complete content")

💾 Chunks saved to: text_chunks_output.txt
🔍 You can inspect this file to see ALL FULL chunks in complete detail!
✅ NO TRUNCATION - Every chunk shows its complete content


## 🧮 Step 4: Generate Embeddings

Embeddings convert text into numerical vectors that capture semantic meaning. Similar texts will have similar vectors.

In [14]:
def get_embedding(text: str) -> List[float]:
    """
    🧮 Get embedding vector for text using OpenAI
    
    What are embeddings?
    - Numerical representations of text meaning
    - Similar texts = similar vectors
    - Enable semantic search (not just keyword matching)
    """
    try:
        response = client.embeddings.create(
            input=text,
            model=RAGConfig.EMBEDDING_MODEL
        )
        return response.data[0].embedding
    except Exception as e:
        print(f"❌ Error getting embedding: {e}")
        return []

print("✅ Embedding function ready!")
print(f"🧮 Using model: {RAGConfig.EMBEDDING_MODEL}")

✅ Embedding function ready!
🧮 Using model: text-embedding-ada-002


In [15]:
# 🧮 Generate embeddings for all chunks
print(f"🧮 Generating embeddings for {len(chunks_with_metadata)} chunks...")
print("⏳ This may take a moment with the OpenAI API...")

successful_embeddings = 0

for i, chunk in enumerate(chunks_with_metadata):
    print(f"Processing chunk {i+1}/{len(chunks_with_metadata)}...", end=" ")
    
    embedding = get_embedding(chunk["text"])
    if embedding:
        chunk["embedding"] = embedding
        chunk["embedding_size"] = len(embedding)
        successful_embeddings += 1
        print("✅")
    else:
        print("❌")
        chunk["embedding"] = []
        chunk["embedding_size"] = 0

print(f"\n📊 Embedding Results:")
print(f"✅ Successful: {successful_embeddings}/{len(chunks_with_metadata)} chunks")
if successful_embeddings > 0:
    print(f"📏 Vector size: {chunks_with_metadata[0]['embedding_size']} dimensions")

🧮 Generating embeddings for 19 chunks...
⏳ This may take a moment with the OpenAI API...
Processing chunk 1/19... ✅
Processing chunk 2/19... ✅
Processing chunk 3/19... ✅
Processing chunk 4/19... ✅
Processing chunk 5/19... ✅
Processing chunk 6/19... ✅
Processing chunk 7/19... ✅
Processing chunk 8/19... ✅
Processing chunk 9/19... ✅
Processing chunk 10/19... ✅
Processing chunk 11/19... ✅
Processing chunk 12/19... ✅
Processing chunk 13/19... ✅
Processing chunk 14/19... ✅
Processing chunk 15/19... ✅
Processing chunk 16/19... ✅
Processing chunk 17/19... ✅
Processing chunk 18/19... ✅
Processing chunk 19/19... ✅

📊 Embedding Results:
✅ Successful: 19/19 chunks
📏 Vector size: 1536 dimensions


## 🗄️ Step 5: Create Vector Database

Let's save our embeddings to a JSON file - our simple vector database!

In [18]:
# 🗄️ Create vector database structure
vector_data = {
    "metadata": {
        "source_document": RAGConfig.INPUT_DOCUMENT,
        "chunk_size": RAGConfig.CHUNK_SIZE,
        "chunk_overlap": RAGConfig.CHUNK_OVERLAP,
        "embedding_model": RAGConfig.EMBEDDING_MODEL,
        "total_chunks": len(chunks_with_metadata),
        "created_at": "2025-09-06"
    },
    "chunks": []
}

# Add only chunks with successful embeddings
for chunk in chunks_with_metadata:
    if chunk.get("embedding"):
        vector_data["chunks"].append({
            "id": chunk["id"],
            "text": chunk["text"],
            "length": chunk["length"],
            "start_pos": chunk["start_pos"],
            "embedding": chunk["embedding"]
        })

print(f"🗄️ Vector database structure created")
print(f"📊 Ready to store {len(vector_data['chunks'])} chunks with embeddings")

🗄️ Vector database structure created
📊 Ready to store 19 chunks with embeddings


In [19]:
# 💾 Save vector database to JSON file
with open(RAGConfig.VECTOR_DATABASE_FILE, 'w', encoding='utf-8') as f:
    json.dump(vector_data, f, indent=2, ensure_ascii=False)

file_size_kb = os.path.getsize(RAGConfig.VECTOR_DATABASE_FILE) / 1024

print(f"💾 Vector database saved to: {RAGConfig.VECTOR_DATABASE_FILE}")
print(f"📊 Stored {len(vector_data['chunks'])} chunks with embeddings")
print(f"💾 File size: {file_size_kb:.1f} KB")

# Show database structure
print(f"\n📋 Database Structure:")
print(f"📁 Metadata keys: {list(vector_data['metadata'].keys())}")
print(f"🔢 Chunks stored: {len(vector_data['chunks'])}")
if vector_data['chunks']:
    print(f"🧮 Embedding dimensions: {len(vector_data['chunks'][0]['embedding'])}")

💾 Vector database saved to: vector_database.json
📊 Stored 19 chunks with embeddings
💾 File size: 886.8 KB

📋 Database Structure:
📁 Metadata keys: ['source_document', 'chunk_size', 'chunk_overlap', 'embedding_model', 'total_chunks', 'created_at']
🔢 Chunks stored: 19
🧮 Embedding dimensions: 1536


## 🔍 Step 6: Similarity Search Implementation

Now the core of RAG - finding relevant chunks using cosine similarity!

In [20]:
def cosine_similarity(vec1: List[float], vec2: List[float]) -> float:
    """
    🧮 Calculate cosine similarity between two vectors
    
    Cosine similarity measures the angle between vectors:
    - 1.0 = identical direction (most similar)
    - 0.0 = perpendicular (unrelated)
    - -1.0 = opposite direction (most dissimilar)
    """
    vec1_np = np.array(vec1)
    vec2_np = np.array(vec2)
    
    # Calculate dot product and magnitudes
    dot_product = np.dot(vec1_np, vec2_np)
    magnitude1 = np.linalg.norm(vec1_np)
    magnitude2 = np.linalg.norm(vec2_np)
    
    # Avoid division by zero
    if magnitude1 == 0 or magnitude2 == 0:
        return 0.0
    
    return dot_product / (magnitude1 * magnitude2)

print("✅ Cosine similarity function ready!")
print("🧮 This measures how similar two vectors are (0.0 to 1.0)")

✅ Cosine similarity function ready!
🧮 This measures how similar two vectors are (0.0 to 1.0)


In [21]:
def search_similar_chunks(question: str, top_k: int = None) -> List[Dict[str, Any]]:
    """
    🔍 Find most similar chunks to the question
    
    Process:
    1. Convert question to embedding
    2. Compare with all chunk embeddings using cosine similarity  
    3. Return top-k most similar chunks
    """
    if top_k is None:
        top_k = RAGConfig.TOP_K_RESULTS
    
    print(f"🔍 Searching for: '{question}'")
    
    # Get question embedding
    question_embedding = get_embedding(question)
    if not question_embedding:
        print("❌ Failed to get question embedding")
        return []
    
    # Calculate similarities with all chunks
    results = []
    for chunk in vector_data["chunks"]:
        if not chunk.get("embedding"):
            continue
            
        similarity = cosine_similarity(question_embedding, chunk["embedding"])
        results.append({
            "chunk_id": chunk["id"],
            "text": chunk["text"],
            "similarity": similarity,
            "length": chunk["length"]
        })
    
    # Sort by similarity (highest first) and take top-k
    results.sort(key=lambda x: x["similarity"], reverse=True)
    top_results = results[:top_k]
    
    print(f"📊 Found {len(results)} chunks, returning top {len(top_results)}")
    
    return top_results

print("✅ Search function ready!")

✅ Search function ready!


In [29]:
# 🧪 Test the search function
test_question = "Which endpoints allows me to find the citizens of bedrock?"
search_results = search_similar_chunks(test_question)

print(f"\n🎯 Top results for: '{test_question}'")
for i, result in enumerate(search_results, 1):
    print(f"\n{i}. Similarity: {result['similarity']:.3f}")
    print(f"   Chunk ID: {result['chunk_id']}")
    print(f"   Preview: {result['text'][:100]}...")

# Save search results to file
import time
timestamp = int(time.time() * 10000) % 10000
filename = f"retrieval_results_{timestamp}.txt"

with open(filename, "w", encoding='utf-8') as f:
    f.write("=" * 60 + "\n")
    f.write("RETRIEVAL RESULTS\n")
    f.write(f"Question: {test_question}\n")
    f.write(f"Top-K results: {len(search_results)}\n")
    f.write("=" * 60 + "\n\n")
    
    for i, result in enumerate(search_results, 1):
        f.write(f"RETRIEVED CHUNK {i}:\n")
        f.write(f"Chunk ID: {result['chunk_id']}\n")
        f.write(f"Similarity Score: {result['similarity']:.6f}\n")
        f.write(f"Text Length: {result['length']} characters\n")
        f.write("-" * 40 + "\n")
        f.write(result['text'])
        f.write("\n" + "=" * 60 + "\n\n")

print(f"\n💾 Search results saved to: {filename}")

🔍 Searching for: 'Which endpoints allows me to find the citizens of bedrock?'
📊 Found 19 chunks, returning top 3

🎯 Top results for: 'Which endpoints allows me to find the citizens of bedrock?'

1. Similarity: 0.821
   Chunk ID: 0
   Preview: api_name:
  Flintstone API
version:
  2.1.0
description:
  The comprehensive API for managing prehis...

2. Similarity: 0.807
   Chunk ID: 2
   Preview: -Dabba-Doo!
            bowling_average:
              165.4
            created_at:
              S...

3. Similarity: 0.801
   Chunk ID: 3
   Preview: v2/citizens/{citizen_id}:
      GET:
        description:
          Get detailed information about a...

💾 Search results saved to: retrieval_results_4267.txt


## 🤖 Step 7: Answer Generation

Finally, let's use the retrieved chunks to generate contextual answers!

In [27]:
def generate_answer(question: str, context_chunks: List[Dict[str, Any]]) -> str:
    """
    🤖 Generate answer using retrieved context
    
    This combines the retrieved chunks with the user's question
    to generate a contextual, accurate response.
    """
    # Combine context from top chunks
    context = "\n\n".join([chunk["text"] for chunk in context_chunks])
    
    # Create prompt with context and question
    prompt = f"""You are a helpful assistant answering questions about the Flintstone API documentation.

Use the following context to answer the question. Be specific and accurate.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""
    
    try:
        response = client.chat.completions.create(
            model=RAGConfig.GENERATION_MODEL,
            messages=[
                {"role": "user", "content": prompt}
            ],
            max_tokens=RAGConfig.MAX_TOKENS,
            temperature=RAGConfig.TEMPERATURE
        )
        
        answer = response.choices[0].message.content.strip()
        return answer
        
    except Exception as e:
        return f"❌ Error generating answer: {e}"

print("✅ Answer generation function ready!")
print(f"🤖 Using model: {RAGConfig.GENERATION_MODEL}")

✅ Answer generation function ready!
🤖 Using model: gpt-4o-mini


In [30]:
# 🧪 Test answer generation with our search results
print("🤖 Generating answer using retrieved chunks...")

answer = generate_answer(test_question, search_results)

print(f"\n🎯 COMPLETE RAG RESULT:")
print("=" * 50)
print(f"❓ Question: {test_question}")
print(f"\n✅ Answer:\n{answer}")
print(f"\n📊 Used {len(search_results)} chunks with similarities: {[round(r['similarity'], 3) for r in search_results]}")

print("\n💾 All outputs have been saved to files for your inspection!")

🤖 Generating answer using retrieved chunks...

🎯 COMPLETE RAG RESULT:
❓ Question: Which endpoints allows me to find the citizens of bedrock?

✅ Answer:
The endpoint that allows you to find the citizens of Bedrock is:

- **GET** `/api/v2/citizens`: This endpoint retrieves all citizens of Bedrock and allows for optional filtering by family name, occupation, and neighborhood.

📊 Used 3 chunks with similarities: [np.float64(0.821), np.float64(0.807), np.float64(0.801)]

💾 All outputs have been saved to files for your inspection!


## 🎮 Step 8: Interactive Exploration

Try asking your own questions! Use the function below to explore the RAG system.

In [25]:
# 🎮 Try your own questions here!
# Change this question and run the cell to test different queries

your_question = "What dinosaur services are available?"

print(f"🎯 Processing your question: '{your_question}'")
print("=" * 50)

# Step 1: Search for similar chunks
print("🔍 Searching for relevant chunks...")
search_results = search_similar_chunks(your_question)

# Step 2: Generate answer
print("🤖 Generating answer...")
answer = generate_answer(your_question, search_results)

# Display results
print(f"\n🎯 YOUR RAG RESULT:")
print("=" * 50)
print(f"❓ Question: {your_question}")
print(f"\n✅ Answer:\n{answer}")
print(f"\n📊 Retrieved {len(search_results)} chunks")
print(f"🔢 Similarity scores: {[round(r['similarity'], 3) for r in search_results]}")

print("\n💾 Check these files for detailed outputs:")
print("  • converted_text_output.txt - Original JSON converted to text")
print("  • text_chunks_output.txt - All text chunks")
print("  • retrieval_results_*.txt - Retrieved chunks with similarity scores")
print("  • vector_database.json - Complete vector database")

🎯 Processing your question: 'What dinosaur services are available?'
🔍 Searching for relevant chunks...
🔍 Searching for: 'What dinosaur services are available?'
📊 Found 19 chunks, returning top 3
🤖 Generating answer...

🎯 YOUR RAG RESULT:
❓ Question: What dinosaur services are available?

✅ Answer:
The available dinosaur services in the Flintstone API documentation are:
- Registering a new working dinosaur
- Renting a dinosaur for temporary work

📊 Retrieved 3 chunks
🔢 Similarity scores: [np.float64(0.814), np.float64(0.794), np.float64(0.773)]

💾 Check these files for detailed outputs:
  • converted_text_output.txt - Original JSON converted to text
  • text_chunks_output.txt - All text chunks
  • retrieval_results_*.txt - Retrieved chunks with similarity scores
  • vector_database.json - Complete vector database


## 🎉 Congratulations! You Built RAG from Scratch!

### 🧠 **What You've Learned:**

1. **📄 Document Processing** - Converting JSON to searchable text
2. **✂️ Text Chunking** - Breaking documents into manageable pieces with overlap
3. **🧮 Embeddings** - Converting text to numerical vectors that capture meaning
4. **🗄️ Vector Database** - Storing embeddings in a simple JSON format
5. **🔍 Similarity Search** - Using cosine similarity to find relevant content
6. **🤖 Answer Generation** - Combining retrieval with AI to create responses
7. **🔬 Transparency** - Saving all outputs to files for inspection

### 🔧 **Key RAG Components:**

- **R**etrieval: Finding relevant chunks using vector similarity
- **A**ugmented: Enhancing the AI with retrieved context  
- **G**eneration: Creating answers based on relevant information

### 🚀 **Next Steps to Explore:**

1. **📚 Add More Documents** - Try different file formats (PDF, markdown, text)
2. **⚙️ Tune Parameters** - Experiment with chunk size, overlap, and top-k results
3. **🔄 Different Models** - Try other embedding models or GPT-4
4. **🎯 Better Chunking** - Implement semantic chunking or paragraph-based splitting
5. **📊 Evaluation** - Add metrics to measure retrieval and generation quality

### 💡 **Understanding the Magic:**

- **Embeddings** transform text into numbers that represent meaning
- **Cosine similarity** finds the "angle" between vectors (closer angle = more similar)
- **Context injection** gives the AI relevant information to generate better answers
- **Chunk overlap** ensures important information doesn't get split

You now understand the fundamentals of RAG and can build upon this foundation! 🎓